In [1]:
import sys, json, glob, os
import numpy as np
import pandas as pd
sys.path.append("../src")
from labels import apply_labels

RAW_DIR = "../data/raw"

rows = []
for p in glob.glob(f"{RAW_DIR}/*.jsonl"):
    with open(p, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass

df = pd.DataFrame(rows)
df["created_at"] = pd.to_datetime(df["created_at"])
df["closed_at"] = pd.to_datetime(df["closed_at"])
print(df.shape)
df["repo"].value_counts()

(32000, 14)


repo
huggingface/transformers     8000
microsoft/vscode             8000
pandas-dev/pandas            8000
scikit-learn/scikit-learn    8000
Name: count, dtype: int64

In [2]:
df = apply_labels(df)
print(df["issue_type"].value_counts(dropna=False))
print(df.groupby(["repo", "issue_type"]).size().unstack(fill_value=0))
print("internal items per repo:\n", df["is_internal"].groupby(df["repo"]).sum())
print("labeled duplicates per repo:\n", df["is_duplicate_labeled"].groupby(df["repo"]).sum())

issue_type
NaN         16041
bug          9523
feature      4056
docs         1906
question      474
Name: count, dtype: int64
issue_type                  bug  docs  feature  question
repo                                                    
huggingface/transformers   2568    15     1120         8
microsoft/vscode           1409     0      531        14
pandas-dev/pandas          3923   878     1055       345
scikit-learn/scikit-learn  1623  1013     1350       107
internal items per repo:
 repo
huggingface/transformers       0
microsoft/vscode             461
pandas-dev/pandas              0
scikit-learn/scikit-learn      0
Name: is_internal, dtype: int64
labeled duplicates per repo:
 repo
huggingface/transformers       1
microsoft/vscode             198
pandas-dev/pandas            176
scikit-learn/scikit-learn      0
Name: is_duplicate_labeled, dtype: int64


In [3]:
for t in ["bug", "feature", "docs"]:
    print("=====", t)
    print(df[df["issue_type"] == t].sample(8, random_state=1)[["repo", "title", "labels"]].to_string())

===== bug
                           repo                                                                                       title                               labels
17024         pandas-dev/pandas                            BUG: Intersection of Pandas Index Object is not working properly                  [Bug, Needs Triage]
8477           microsoft/vscode                                remote AH entries in recent folders do not show their remote                    [bug, agent-host]
17503         pandas-dev/pandas                                          BUG: `Timestamp.normalize()` can overflow silently                     [Bug, Timestamp]
2537   huggingface/transformers                 Trainer.training_step incorrectly normalizes mean token loss when n_gpu > 1              [Good First Issue, bug]
20621         pandas-dev/pandas                                            BUG: Bad shift_forward ceiling around DST change          [Bug, Timezones, Timestamp]
1467   huggingface/trans

In [4]:
df["text"] = df["title"].fillna("") + "\n\n" + df["body"].fillna("")

is_bot = df["author"].fillna("").str.endswith("[bot]") | df["author"].eq("Copilot")
is_vscode_team = (df["repo"] == "microsoft/vscode") & df["author_association"].isin(["MEMBER", "COLLABORATOR", "OWNER"])
df["exclude"] = df["is_internal"] | is_bot | is_vscode_team

print(df.groupby("repo")["exclude"].agg(["sum", "mean"]).round(2))
df_all = df[~df["exclude"]].copy()          # all real user issues (also used for duplicate search later)
print("all real issues:", len(df_all), " with a type:", df_all["issue_type"].notna().sum())
df_all.dropna(subset=["issue_type"]).groupby(["repo", "issue_type"]).size().unstack(fill_value=0)

                            sum  mean
repo                                 
huggingface/transformers     98  0.01
microsoft/vscode           2700  0.34
pandas-dev/pandas             1  0.00
scikit-learn/scikit-learn     0  0.00
all real issues: 29201  with a type: 14878


issue_type,bug,docs,feature,question
repo,,,,
huggingface/transformers,2568,15,1120,8
microsoft/vscode,638,0,222,14
pandas-dev/pandas,3923,877,1055,345
scikit-learn/scikit-learn,1623,1013,1350,107


In [5]:
# only issues old enough to have collected engagement
cutoff = df_all["created_at"].max() - pd.Timedelta(days=14)
old = df_all[df_all["created_at"] < cutoff].copy()

old["engagement"] = np.log1p(old["comments"]) + 2 * np.log1p(old["reactions_total"])
old["eng_pct"] = old.groupby("repo")["engagement"].rank(pct=True)   # rank inside each repo
old["priority"] = pd.cut(old["eng_pct"], bins=[0, 0.5, 0.85, 1.0],
                         labels=["low", "medium", "high"], include_lowest=True)

df_all = df_all.merge(old[["repo", "number", "engagement", "priority"]],
                      on=["repo", "number"], how="left")
print(df_all["priority"].value_counts(dropna=False))
df_all.groupby("repo")["priority"].value_counts(normalize=True).unstack().round(2)

priority
low       14702
medium     9486
high       4327
NaN         686
Name: count, dtype: int64


priority,low,medium,high
repo,,,
huggingface/transformers,0.55,0.31,0.15
microsoft/vscode,0.51,0.33,0.16
pandas-dev/pandas,0.54,0.31,0.16
scikit-learn/scikit-learn,0.47,0.38,0.15


In [6]:
parts = []
for repo, g in df_all.groupby("repo"):
    g = g.sort_values("created_at").copy()
    n = len(g); a, b = int(n * 0.70), int(n * 0.85)
    g["split"] = ["train"] * a + ["val"] * (b - a) + ["test"] * (n - b)
    parts.append(g)
df_all = pd.concat(parts, ignore_index=True)

# task subsets (derived when needed, not saved separately)
typed    = df_all.dropna(subset=["issue_type"])
tag_pool = df_all[df_all["labels"].apply(len) > 0]
prio     = df_all.dropna(subset=["priority"])

print(df_all.groupby(["repo", "split"])["created_at"].agg(["min", "max"]))   # test must be newest in every repo
for name, d in [("typed", typed), ("tag_pool", tag_pool), ("prio", prio)]:
    print(name, d.groupby("split").size().to_dict())
print(typed.groupby(["split", "issue_type"]).size().unstack())
print(tag_pool.explode("tags").groupby(["split", "tags"]).size().unstack(0))

                                                      min  \
repo                      split                             
huggingface/transformers  test  2025-12-05 16:10:04+00:00   
                          train 2023-05-12 09:08:23+00:00   
                          val   2025-04-24 09:02:24+00:00   
microsoft/vscode          test  2026-09-15 18:18:15+00:00   
                          train 2026-07-02 23:00:19+00:00   
                          val   2026-09-03 03:04:20+00:00   
pandas-dev/pandas         test  2025-05-02 16:28:12+00:00   
                          train 2021-05-24 14:42:08+00:00   
                          val   2024-03-29 07:02:58+00:00   
scikit-learn/scikit-learn test  2024-09-06 08:33:52+00:00   
                          train 2017-09-08 16:13:01+00:00   
                          val   2023-02-03 20:17:51+00:00   

                                                      max  
repo                      split                            
huggingface/transformers 

In [7]:
os.makedirs("../data/processed", exist_ok=True)
df_all["priority"] = df_all["priority"].astype("object")
df_all.to_parquet("../data/processed/issues_all.parquet", index=False)

check = pd.read_parquet("../data/processed/issues_all.parquet")
print("saved:", check.shape)
print([c for c in ["issue_type", "tags", "priority", "split", "text"] if c not in check.columns], "<- missing columns (should be empty)")

saved: (29201, 23)
[] <- missing columns (should be empty)


In [8]:
# VS Code composition after filtering
v = df_all[df_all["repo"] == "microsoft/vscode"]
print(v["author_association"].value_counts(normalize=True).round(2))
print(v["author"].value_counts().head(8))
print("share with no labels:", v["labels"].apply(len).eq(0).mean().round(2))
print(v["created_at"].dt.tz_localize(None).dt.to_period("M").value_counts().sort_index())

# issues closed as duplicates (ground truth candidates for Day 5)
print(df_all["state_reason"].value_counts(dropna=False))
print(df_all[df_all["state_reason"] == "duplicate"].groupby("repo").size())

# any hidden question-type labels?
ll = df.explode("labels")
print(ll[ll["labels"].str.contains("question|usage|support", case=False, na=False)].groupby(["repo", "labels"]).size())

author_association
NONE           0.8
CONTRIBUTOR    0.2
Name: proportion, dtype: float64
author
ulugbekna       176
benibenj         87
digitarald       72
aeschli          68
jruales          64
anthonykim1      61
DonJayamanne     45
mrleemurray      44
Name: count, dtype: int64
share with no labels: 0.46
created_at
2026-07    1733
2026-08    1845
2026-09    1635
2026-10      87
Freq: M, Name: count, dtype: int64
state_reason
completed      21118
NaN             5535
not_planned     1744
duplicate        683
reopened         121
Name: count, dtype: int64
repo
huggingface/transformers      14
microsoft/vscode             634
pandas-dev/pandas             22
scikit-learn/scikit-learn     13
dtype: int64
repo                       labels                
huggingface/transformers   Ex: Question Answering      1
                           Usage                      55
microsoft/vscode           *question                  14
pandas-dev/pandas          Usage Question            364
scikit-l